In [1]:
import os
import ffmpeg
import subprocess
import json
from tqdm import tqdm
import time
import math 

In [4]:
concactenatedFilePath=r"2024_12_30\994_18_25_08\My_WebCam\m994_30122024_18_25_08_concactenatedbehavCam00_behavCam17.mp4"
angleToRotate = 30
linearTrackPath = r"R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track"

video_file = os.path.join(
    linearTrackPath,
    concactenatedFilePath
)
folder = os.path.dirname(video_file)
filename = os.path.basename(video_file)
stem, ext = os.path.splitext(filename)
output_file = os.path.join(
    folder,
    stem + "_rotated_angletest_padded" + ext
)
print(video_file)
print(output_file)

angleToRotate = 30

R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\m994_30122024_18_25_08_concactenatedbehavCam00_behavCam17.mp4
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\m994_30122024_18_25_08_concactenatedbehavCam00_behavCam17_rotated_angletest_padded.mp4


In [ ]:
# Use ffmpeg to retrieve the metadata
try:
    # Run ffmpeg to get the file's metadata
    probe = ffmpeg.probe(video_file)
    
    # Extract the codec type for the video stream
    video_stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'video'), None)
    
    if video_stream:
        codec_name = video_stream.get('codec_name', 'Unknown')
        print(f"Video codec: {codec_name}")
    else:
        print("No video stream found.")
        
except ffmpeg.Error as e:
    print("An error occurred while reading the video file:", e)

input_file = video_file

# Get the video dimensions and duration
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])
duration = float(video_info['duration'])

# Calculate new dimensions to fit the rotated video
angle_rad = math.radians(angleToRotate)
new_width = int(abs(width * math.cos(angle_rad)) + abs(height * math.sin(angle_rad)))
new_height = int(abs(width * math.sin(angle_rad)) + abs(height * math.cos(angle_rad)))

# Define the ffmpeg command to add padding, rotate, and output with JPEG compression
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('pad', new_width, new_height, (new_width - width) // 2, (new_height - height) // 2, color='0xFFFFFF')
    .filter('rotate', str(angle_rad))
    .output(output_file, vcodec='mjpeg', vsync='vfr')
    .global_args('-progress', 'pipe:1', '-nostats')  # Enable progress output
)

# Run ffmpeg as a subprocess and track progress
process = ffmpeg_command.run_async(pipe_stdout=True, pipe_stderr=True, overwrite_output=True)
pbar = tqdm(total=duration, desc="Processing Video", unit="s", dynamic_ncols=True)

# Track ffmpeg progress
for line in process.stderr:
    line = line.decode('utf-8').strip()
    if "out_time_ms" in line:
        out_time_ms = int(line.split('=')[1].strip())
        current_time = out_time_ms / 1_000_000  # Convert to seconds
        pbar.update(current_time - pbar.n)

pbar.close()
process.wait()
print("Processing complete.")


Video codec: mpeg2video


Processing Video:   0%|                                                 | 0/286.75 [00:00<?, ?s/s]

In [ ]:
#crop video 
# Input and output file paths
input_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20_rotated_30deg_padded.mp4"
output_file = "/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m311/2025_04_02_311_19_15_29_b2/My_WebCam/2025_04_02_311_19_15_29_b2_concactenatedbehavCam00_behavCam20_rotated_cropped_output.avi"
# Define the four vertices of the rectangle for cropping (x1, y1), (x2, y2), (x3, y3), (x4, y4)
vertices = [(51, 373), (709, 373), (709, 404), (51, 404)]


# Get the video dimensions
probe = ffmpeg.probe(input_file)
video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
width = int(video_info['width'])
height = int(video_info['height'])

# Calculate the bounding box for cropping
x_coordinates = [v[0] for v in vertices]
y_coordinates = [v[1] for v in vertices]
x_min, x_max = max(0, min(x_coordinates)), min(width, max(x_coordinates))
y_min, y_max = max(0, min(y_coordinates)), min(height, max(y_coordinates))

# Calculate crop dimensions
crop_x = x_min
crop_y = y_min
crop_width = x_max - x_min
crop_height = y_max - y_min

# Validate crop dimensions to ensure they are within the video frame
if crop_width <= 0 or crop_height <= 0 or crop_x + crop_width > width or crop_y + crop_height > height:
    raise ValueError(f"Invalid crop dimensions: {crop_width}x{crop_height} at position ({crop_x}, {crop_y}). "
                     f"Ensure crop area is within video bounds {width}x{height}.")

# Define the ffmpeg command to crop the video
ffmpeg_command = (
    ffmpeg.input(input_file)
    .filter('crop', crop_width, crop_height, crop_x, crop_y)
    .output(output_file, vcodec='libx264', crf=23, pix_fmt='yuv420p')  # Adjust codec/compression as needed
)

# Run ffmpeg command
ffmpeg_command.run(overwrite_output=True)
print("Cropping complete.")